# Path B-lite Step 1: Feature extraction + sanity diagnostic

**Date:** 2026-04-18.
**Goal:** Verify each proposed Path B-lite feature (a) has a defined cohort distribution we can tune sigma to, and (b) correlates with `actual_phase1` enough to plausibly help rate-matching selection.

**Features computed per target at T-3d:**
- `gap` — already in combined_score
- `rate` — critics per day in observed window (already tested in B-0)
- `top_critic_frac` — fraction of observed reviews marked top_critic
- `low_activity_critic_frac` — fraction of observed critics whose cohort-wide activity < 5 (half of critics cohort-wide have ≤4 reviews)
- `pub_diversity` — unique publications in observed set
- `pub_entropy` — Shannon entropy over publication frequencies

**Question for each feature:** does it correlate with `actual_phase1`? If Q4 high-vol targets have systematically different values than Q1 low-vol targets on a feature, rate-matching on that feature can plausibly help.

**Note:** This step only computes TARGET feature values, not per-(target, candidate) similarities. That's Step 2.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name != 'notebooks':
    ROOT = ROOT / 'notebooks'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from _helpers import (
    reviews, close_date_map, gap_lookup, first_review_ts, gap_for_slug,
    snapshot_state, passes_skip_rules_for_snap,
    critic_activity_counts, observed_review_stats,
    CACHE_DIR,
)

SNAP = 3.0
CACHE = CACHE_DIR / 'path_b_lite_step1_features.pkl'

# Pre-compute critic activity across the whole cohort (defines who's a "low-activity" critic)
activity = critic_activity_counts()
print(f'Total critics in cohort: {len(activity)}')
act_ser = pd.Series(activity)
print(f'Activity percentiles: {dict(act_ser.describe([.1, .25, .5, .75, .9]).round(0))}')
print()

# Threshold for "low-activity": critics with fewer than N reviews in the cohort
LOW_ACTIVITY_THRESHOLD = 5
pct_low = (act_ser < LOW_ACTIVITY_THRESHOLD).mean() * 100
print(f'Low-activity threshold = {LOW_ACTIVITY_THRESHOLD} → {pct_low:.0f}% of critics qualify as "low-activity"')

## Compute feature values per target at T-3d

In [ ]:
def compute_target_features(force=False):
    if CACHE.exists() and not force:
        return pd.read_pickle(CACHE)

    rows = []
    for target in close_date_map:
        target_gap = gap_for_slug(target)
        if target_gap is None:
            continue
        target_close = close_date_map[target]
        midnight_utc_dbc = (target_close - target_close.floor('D')).total_seconds() / 86400
        snap_time = target_close - pd.Timedelta(days=SNAP)
        state = snapshot_state(target, snap_time)
        passed, _ = passes_skip_rules_for_snap(state, SNAP)
        if not passed:
            continue

        target_first_review = first_review_ts.loc[target]
        target_window_days = state['first_review_dbc'] - SNAP
        if target_window_days <= 0:
            continue

        # Compute target's observed-window feature values
        stats = observed_review_stats(
            target, target_first_review, target_window_days, activity,
            low_activity_threshold=LOW_ACTIVITY_THRESHOLD,
        )

        # Ground truth phase 1
        movie_reviews = reviews[reviews['movie_slug'] == target].copy()
        movie_reviews['dbc'] = (target_close - movie_reviews['estimated_timestamp']).dt.total_seconds() / 86400
        actual_p1 = int(((movie_reviews['dbc'] > midnight_utc_dbc) & (movie_reviews['dbc'] <= SNAP)).sum())

        rows.append({
            'target': target,
            'target_gap': target_gap,
            'target_window_days': target_window_days,
            'observed_count': state['observed_count'],
            **stats,
            'actual_phase1': actual_p1,
        })

    df = pd.DataFrame(rows)
    df.to_pickle(CACHE)
    print(f'Cached {len(df)} rows')
    return df

feat = compute_target_features()
print(f'n={len(feat)}')
print()
print('Target feature distributions:')
cols = ['target_gap', 'rate', 'top_critic_frac', 'low_activity_frac', 'pub_diversity', 'pub_entropy']
print(feat[cols].describe([.1, .25, .5, .75, .9]).round(2).to_string())

## Pick sigma per feature (IQR/2) for similarity functions

In [ ]:
sigmas = {}
print('Sigma choices for similarity features (exp(-|diff|/sigma)):')
for col in ['target_gap', 'rate', 'top_critic_frac', 'low_activity_frac',
            'pub_diversity', 'pub_entropy']:
    iqr = feat[col].quantile(0.75) - feat[col].quantile(0.25)
    sigma = round(iqr / 2, 3)
    sigmas[col] = sigma
    print(f'  {col:25s}  IQR={iqr:7.3f}  sigma={sigma}')
print()
print('These sigmas will be plugged into exp(-|target_feat - candidate_feat| / sigma) at Step 2.')

## Correlation between each feature and actual_phase1

If a feature has near-zero correlation with actual_phase1, it cannot discriminate high-vol from low-vol targets and should be dropped before Step 2.

In [ ]:
from scipy.stats import spearmanr, pearsonr

print('Correlation of target features with actual_phase1:')
print(f'{"feature":25s}  {"pearson_r":>10s}  {"spearman_r":>11s}')
for col in ['target_gap', 'rate', 'top_critic_frac', 'low_activity_frac',
            'pub_diversity', 'pub_entropy', 'observed_count']:
    p_r, _ = pearsonr(feat[col], feat['actual_phase1'])
    s_r, _ = spearmanr(feat[col], feat['actual_phase1'])
    print(f'  {col:25s}  {p_r:+10.3f}  {s_r:+11.3f}')

## Stratified feature means (by actual_phase1 quartile)

For each feature, show how the cohort mean differs across volume quartiles. A useful feature should have monotonic-ish differences Q1→Q4.

In [ ]:
feat['q_actual'] = pd.qcut(feat['actual_phase1'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop')

summary_cols = ['rate', 'top_critic_frac', 'low_activity_frac', 'pub_diversity', 'pub_entropy']
print('Feature means by actual_phase1 quartile:')
print(f'{"":28s}  ' + '  '.join(f'{q:>10s}' for q in ['Q1', 'Q2', 'Q3', 'Q4']))
print(f'{"(actual range)":28s}  ', end='')
for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    sub = feat[feat['q_actual'] == q]
    lo, hi = int(sub['actual_phase1'].min()), int(sub['actual_phase1'].max())
    print(f'{"[" + str(lo) + "-" + str(hi) + "]":>10s}  ', end='')
print()
print()
for col in summary_cols:
    print(f'  {col:25s}  ', end='')
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        sub = feat[feat['q_actual'] == q]
        print(f'{sub[col].mean():10.3f}  ', end='')
    print()

## H/m subset inspection

In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_feat = feat[feat['target'].isin(HM)].copy()
cols = ['target', 'target_gap', 'rate', 'top_critic_frac',
        'low_activity_frac', 'pub_diversity', 'pub_entropy', 'actual_phase1']
print('H/m subset features:')
print(hm_feat[cols].to_string(index=False, float_format='%.3f'))

## Decision

After reading the output:

1. **Features with weak correlation (|r| < 0.2) on both pearson and spearman** → drop before Step 2.
2. **Features with strong monotonic Q1→Q4 pattern** → high-priority for rate-matching.
3. **Sigmas recorded above** get used in Step 2's similarity functions.